# Pipeline 2 — Reintegration Drivers (Explanatory)

## 1) Problem Framing

**Business question:** Which factors most strongly *explain* reintegration completion (not only predict it), and in what direction?

- **Type:** Explanatory / inference-oriented (interpretable coefficients).
- **Target:** `reintegration_complete` — same definition as Pipeline 1 (`reintegration_status == 'Completed'`).
- **Primary metrics:** Adjusted R² (OLS), coefficient significance (p-values), pseudo-R² (logistic comparison).
- **Operational use:** Publish **one org-level insight** row to `ml_predictions` with top drivers for impact/donor messaging.

### Prediction vs explanation (textbook framing)

Pipeline 1 prioritizes **predictive accuracy** (ROC-AUC, held-out discrimination). Pipeline 2 prioritizes **interpretable associations** (linear coefficients on a defensible feature set, multicollinearity control). The same resident-month features can support both goals, but feature selection and evaluation criteria differ: here we avoid RFECV-for-prediction as the final selector and emphasize VIF and domain reasoning.


> **Environment requirement:** This notebook loads data from the project's Azure PostgreSQL database via shared ETL modules. To run top-to-bottom, you need:
> 1. A `.env` file in the repo root with valid database credentials (see `.env.example`)
> 2. Python packages from `ml/requirements.txt` installed (`pip install -r ml/requirements.txt`)
> 3. Network access to `intex-db.postgres.database.azure.com`
>
> All data preparation and cleaning is handled by the ETL module to ensure reproducibility across pipelines. The missing value check and feature summary below document the state of the data after ETL processing.

In [ ]:
# 2) Data Acquisition and Preparation — import shared features
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from scipy import stats
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.diagnostic import het_breuschpagan

# Ensure imports work regardless of notebook launch directory.
cwd = Path.cwd().resolve()
candidates = [cwd] + list(cwd.parents)
for p in candidates:
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))
for p in candidates:
    ml_path = p / "ml"
    if ml_path.exists() and str(ml_path) not in sys.path:
        sys.path.insert(0, str(ml_path))

try:
    from ml.config import MODEL_REINTEGRATION_DRIVERS, MODEL_RUNS_REINTEGRATION_DRIVERS
    from ml.reintegration_readiness.etl import build_training_frame
    from ml.reintegration_drivers.artifacts import save_metadata, save_metrics, save_model_bundle
except ModuleNotFoundError:
    from config import MODEL_REINTEGRATION_DRIVERS, MODEL_RUNS_REINTEGRATION_DRIVERS
    from reintegration_readiness.etl import build_training_frame
    from reintegration_drivers.artifacts import save_metadata, save_metrics, save_model_bundle

RANDOM_STATE = 42

In [2]:
train_df = build_training_frame()

assert "reintegration_complete" in train_df.columns
assert train_df["resident_id"].is_unique

y = train_df["reintegration_complete"].astype(int)
X = train_df.drop(columns=["reintegration_complete", "resident_id"], errors="ignore")

print("Rows:", len(train_df))
print("Feature count:", X.shape[1])
print("Class distribution:\n", y.value_counts())


Rows: 60
Feature count: 34
Class distribution:
 reintegration_complete
0    41
1    19
Name: count, dtype: int64


In [ ]:
# --- Missing value and outlier check ---
print('=== Missing Values ===')
missing = X.isnull().sum()
if missing.sum() == 0:
    print('No missing values in the feature matrix.')
else:
    print(missing[missing > 0])

print(f'
=== Dataset Shape ===')
print(f'Rows: {len(X)}, Features: {X.shape[1]}')

print(f'
=== Outlier Check (numeric features) ===')
outlier_found = False
for col in X.select_dtypes(include=[np.number]).columns:
    q1, q3 = X[col].quantile(0.25), X[col].quantile(0.75)
    iqr = q3 - q1
    outliers = ((X[col] < q1 - 1.5 * iqr) | (X[col] > q3 + 1.5 * iqr)).sum()
    if outliers > 0:
        print(f'  {col}: {outliers} IQR outliers ({outliers/len(X)*100:.1f}%)')
        outlier_found = True
if not outlier_found:
    print('  No IQR outliers detected in any numeric feature.')

print(f'
=== Feature Summary ===')
display(X.describe(include="all").T)


## 3) Exploration (Ch. 6-8)

This notebook must be self-contained. Below we show key distributions, correlations, and relationships in the data — not deferring to Pipeline 1.

In [ ]:
# 3a) Target distribution
fig, ax = plt.subplots(figsize=(6, 4))
y.value_counts().plot(kind="bar", color=["#3d5a80", "#e07a5f"], ax=ax)
ax.set_xticklabels(["Not Completed (0)", "Completed (1)"], rotation=0)
ax.set_ylabel("Count")
ax.set_title("Reintegration Completion Distribution")
for i, v in enumerate(y.value_counts().values):
    ax.text(i, v + 0.3, str(v), ha="center", fontweight="bold")
plt.tight_layout()
plt.show()

print(f"Completion rate: {y.mean():.1%} ({y.sum()} of {len(y)} residents)")

# 3b) Correlation with target
corr = train_df.drop(columns=["resident_id"], errors="ignore").corr(numeric_only=True)
print("\nTop correlations with reintegration_complete:")
print(corr["reintegration_complete"].sort_values(ascending=False).head(15))

In [ ]:
# 3c) Correlation heatmap
fig, ax = plt.subplots(figsize=(14, 10))
sns.heatmap(corr, cmap="RdBu_r", center=0, ax=ax, linewidths=0.5, square=True)
ax.set_title("Feature Correlation Heatmap")
plt.tight_layout()
plt.show()

# 3d) Key feature distributions by completion status
key_features = ["visits_per_month", "trauma_severity_score", "courses_completed",
                "psych_checkups", "favorable_rate", "positive_session_rate"]
available = [f for f in key_features if f in X.columns]

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()
for i, feat in enumerate(available):
    ax = axes[i]
    for label, color in [(0, "#3d5a80"), (1, "#e07a5f")]:
        subset = X.loc[y == label, feat].dropna()
        ax.hist(subset, bins=10, alpha=0.6, color=color,
                label=f"{'Not Completed' if label==0 else 'Completed'}")
    ax.set_title(feat)
    ax.legend()
for j in range(len(available), len(axes)):
    axes[j].set_visible(False)
plt.suptitle("Key Features by Reintegration Status", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

# 3e) Reintegration rates by case category
case_cols = [c for c in train_df.columns if c.startswith("case_category_")]
if case_cols:
    fig, ax = plt.subplots(figsize=(8, 4))
    rates = {}
    for col in case_cols:
        mask = train_df[col] == 1
        if mask.sum() > 0:
            rates[col.replace("case_category_", "")] = train_df.loc[mask, "reintegration_complete"].mean()
    rate_series = pd.Series(rates).sort_values(ascending=False)
    rate_series.plot(kind="bar", color="#3d5a80", ax=ax)
    ax.set_ylabel("Completion Rate")
    ax.set_title("Reintegration Rate by Case Category")
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45)
    plt.tight_layout()
    plt.show()

## 4) Feature Selection — interpretability & multicollinearity (Ch. 16 style)

Steps (no RFECV as final selector):

1. Drop near-zero variance.
2. Drop pairwise redundant features (|r| > 0.95) on the training split.
3. Iterative **VIF** pruning (drop the highest VIF feature while any VIF > 10) to stabilize coefficients.
4. Optional domain drops: document any removals that are mediators/proxies.

Target **~8–12** interpretable predictors where possible.


In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

# Near-zero variance
variances = X_train.var(numeric_only=True)
keep_var = variances[variances > 1e-12].index.tolist()
X_train_v = X_train[keep_var].copy()
X_test_v = X_test[keep_var].copy()

# High correlation pruning (training only)
corr_mat = X_train_v.corr(numeric_only=True).abs()
upper = corr_mat.where(np.triu(np.ones(corr_mat.shape), k=1).astype(bool))
drop_corr = [col for col in upper.columns if any(upper[col] > 0.95)]
X_train_f = X_train_v.drop(columns=drop_corr, errors="ignore")
X_test_f = X_test_v.drop(columns=drop_corr, errors="ignore")
print("After variance + correlation pruning:", X_train_f.shape[1], "features")


def compute_vif(frame: pd.DataFrame) -> pd.DataFrame:
    vals = frame.values.astype(float)
    vifs = []
    for i in range(vals.shape[1]):
        vifs.append(variance_inflation_factor(vals, i))
    return pd.DataFrame({"feature": frame.columns, "VIF": vifs})


X_vif = X_train_f.copy()
while True:
    vif_df = compute_vif(X_vif)
    vmax = vif_df["VIF"].replace([np.inf, -np.inf], np.nan).max()
    if np.isnan(vmax) or vmax <= 10:
        break
    worst = vif_df.sort_values("VIF", ascending=False).iloc[0]["feature"]
    X_vif = X_vif.drop(columns=[worst])
    print("Dropped (high VIF):", worst, "max VIF was", vmax)

X_sel = X_vif.copy()
X_test_sel = X_test_f[X_sel.columns]
print("Final feature count:", X_sel.shape[1])
print(compute_vif(X_sel))


After variance + correlation pruning: 29 features
Dropped (high VIF): initial_risk_num max VIF was 557.9573590686064
Dropped (high VIF): case_category_Surrendered max VIF was 557.9573590686064
Dropped (high VIF): avg_attendance max VIF was 477.4799499792898
Dropped (high VIF): avg_health max VIF was 138.14328205998137
Dropped (high VIF): checkup_compliance max VIF was 58.80489732238087
Dropped (high VIF): avg_progress max VIF was 48.173121134665074
Dropped (high VIF): total_visits max VIF was 33.509740280553885
Dropped (high VIF): age_at_admission max VIF was 22.237142651681207
Dropped (high VIF): family_coop_rate max VIF was 19.46498610897635
Dropped (high VIF): medical_checkups max VIF was 14.468434884331282
Dropped (high VIF): visits_per_month max VIF was 13.070444337343455
Dropped (high VIF): length_of_stay_days max VIF was 11.934061398948971
Final feature count: 17
                       feature       VIF
0             current_risk_num  7.663096
1               risk_reduction  2.0

C:\Users\justi\AppData\Roaming\Python\Python314\site-packages\statsmodels\stats\outliers_influence.py:197: RuntimeWarning: divide by zero encountered in scalar divide
  vif = 1. / (1. - r_squared_i)


## 5-6) Modeling — OLS (primary), logistic (comparison), tree (sanity check)

- Fit **OLS** on **standardized** training features (coefficients are comparable "per SD").
- Fit **Logit** on the same design matrix for direction/significance comparison on the binary target.
- Fit a shallow **Decision Tree** only as a non-primary sanity check.

### OLS Assumptions (checked below)
1. **Linearity** — assumed with standardized features
2. **Independence** — each resident is a separate observation
3. **Homoscedasticity** — tested via Breusch-Pagan
4. **Normality of residuals** — tested via Shapiro-Wilk
5. **No severe multicollinearity** — VIF already checked in feature selection

### Logistic Regression Note
With n=48 training rows and 17 features, logistic regression is severely **overparameterized** (fewer than 3 observations per parameter). This makes perfect separation likely, causing the model to fail with a singularity error. This is a data limitation, not a code bug — it confirms why we use OLS as the primary model for this explanatory pipeline.

In [5]:
scaler = StandardScaler()
X_train_std = scaler.fit_transform(X_sel)
X_test_std = scaler.transform(X_test_sel)

X_train_sm = sm.add_constant(pd.DataFrame(X_train_std, columns=X_sel.columns, index=X_sel.index))
X_test_sm = sm.add_constant(pd.DataFrame(X_test_std, columns=X_sel.columns, index=X_test_sel.index))

ols = sm.OLS(y_train, X_train_sm).fit()
print(ols.summary())

logit = None
try:
    logit = sm.Logit(y_train, X_train_sm).fit(disp=0)
    print("\nLogit pseudo R2 (McFadden):", getattr(logit, "prsquared", np.nan))
    print(logit.summary())
except Exception as e:
    print("\nLogit fit skipped due to model singularity/instability:", e)

tree = DecisionTreeClassifier(max_depth=3, min_samples_leaf=4, random_state=RANDOM_STATE)
tree.fit(X_train_std, y_train)
print("Tree train acc:", tree.score(X_train_std, y_train))
print("Tree test acc:", tree.score(X_test_std, y_test))


                              OLS Regression Results                              
Dep. Variable:     reintegration_complete   R-squared:                       0.604
Model:                                OLS   Adj. R-squared:                  0.379
Method:                     Least Squares   F-statistic:                     2.690
Date:                    Tue, 07 Apr 2026   Prob (F-statistic):            0.00866
Time:                            11:39:35   Log-Likelihood:                -8.9778
No. Observations:                      48   AIC:                             53.96
Df Residuals:                          30   BIC:                             87.64
Df Model:                              17                                         
Covariance Type:                nonrobust                                         
                                 coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------

C:\Users\justi\AppData\Roaming\Python\Python314\site-packages\statsmodels\discrete\discrete_model.py:2385: RuntimeWarning: overflow encountered in exp
  return 1/(1+np.exp(-X))
C:\Users\justi\AppData\Roaming\Python\Python314\site-packages\statsmodels\discrete\discrete_model.py:2443: RuntimeWarning: divide by zero encountered in log
  return np.sum(np.log(self.cdf(q * linpred)))


## Evaluation — explanatory diagnostics

- **Adjusted R²** (OLS) and **pseudo-R²** (logistic).
- **Coefficient table** with p-values and CIs (from OLS summary).
- **VIF** on the final standardized training matrix (flag VIF > 5).
- **Residual checks** (OLS): rough normality of residuals + heteroskedasticity note.


In [ ]:
# Evaluation — OLS diagnostics + cross-validation
print("=" * 70)
print("OLS DIAGNOSTICS")
print("=" * 70)
print(f"Adjusted R2 (train): {ols.rsquared_adj:.4f}")
print(f"R2 (train): {ols.rsquared:.4f}")

# VIF on final model
final_vif = compute_vif(pd.DataFrame(X_train_std, columns=X_sel.columns))
print("\nFinal VIF:")
print(final_vif.to_string(index=False))
print("\nFlags VIF > 5:", final_vif[final_vif["VIF"] > 5]["feature"].tolist())

# Residual diagnostics
resid = ols.resid
fitted = ols.fittedvalues

# Assumption checks
shapiro_stat, shapiro_p = stats.shapiro(resid)
print(f"\nResidual Shapiro-Wilk p (normality): {shapiro_p:.4f}", 
      "PASS" if shapiro_p > 0.05 else "FAIL")

bp_stat, bp_p, _, _ = het_breuschpagan(resid, X_train_sm)
print(f"Breusch-Pagan p (homoscedasticity): {bp_p:.4f}",
      "PASS" if bp_p > 0.05 else "FAIL")

# Residual plots
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Residuals vs fitted
axes[0].scatter(fitted, resid, alpha=0.6, color="#3d5a80")
axes[0].axhline(y=0, color="red", linestyle="--")
axes[0].set_xlabel("Fitted Values")
axes[0].set_ylabel("Residuals")
axes[0].set_title("Residuals vs Fitted")

# Histogram of residuals
axes[1].hist(resid, bins=15, color="#3d5a80", edgecolor="white")
axes[1].set_xlabel("Residual")
axes[1].set_ylabel("Frequency")
axes[1].set_title(f"Residual Distribution (Shapiro p={shapiro_p:.3f})")

# Q-Q plot
stats.probplot(resid, plot=axes[2])
axes[2].set_title("Q-Q Plot of Residuals")

plt.tight_layout()
plt.show()

# Cross-validation of OLS via sklearn wrapper
from sklearn.linear_model import LinearRegression
lr_cv = LinearRegression()
scaler_cv = StandardScaler()
X_all_std = scaler_cv.fit_transform(X[X_sel.columns])
kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_r2 = cross_val_score(lr_cv, X_all_std, y, cv=kf, scoring="r2")
print(f"\n5-fold CV R2: {cv_r2.mean():.4f} (+/- {cv_r2.std():.4f})")
print(f"Per-fold: {cv_r2}")

### Business Interpretation of Evaluation Results

The OLS diagnostics should be interpreted in the context of understanding *what drives successful reintegration*, not predicting individual outcomes:

- **Adjusted R² (~0.38):** About 38% of the variation in reintegration completion is explained by the included features. The remaining 62% reflects unmeasured factors (family dynamics, individual resilience, external circumstances). This is a reasonable explanatory power for behavioral data in a small-sample social services context.
- **Cross-validation R²:** The CV estimate provides a more honest picture of how stable the coefficient estimates are. If CV R² drops substantially below the training R², the model may be overfitting to the small sample (n=60).
- **Residual normality (Shapiro-Wilk):** Passing this test means the standard errors and p-values from the OLS summary are trustworthy — important since we are making inference claims based on coefficient significance.
- **Practical significance:** A coefficient of 0.13 for `psych_checkups` means that a one-standard-deviation increase in psychological check-up compliance is associated with a 13-percentage-point increase in reintegration completion probability. For the organization, this suggests that investing in regular psychological monitoring may yield meaningful improvements in outcomes.
- **The `courses_completed` paradox:** The negative coefficient does not mean education hurts reintegration. It likely reflects confounding — residents in longer educational programs are those with more complex situations requiring extended care. The organization should not reduce educational programming based on this finding.

## 7) Causal and Relationship Analysis

These models estimate **associations** under linear functional forms. They are **not** causal impacts without stronger design.

### Key Findings from OLS Coefficients (standardized)

**Significant or near-significant results:**

- **`courses_completed`** (coef ~ -0.18, p=0.008): Counterintuitively *negative* — residents who completed more educational courses are *less* likely to complete reintegration. Possible explanation: residents in longer-term educational programs may still be mid-program and haven't yet reached the reintegration stage. This could also reflect that educational engagement is a proxy for longer stays in cases where reintegration is harder.

- **`psych_checkups`** (coef ~ +0.13, p=0.053): Marginally positive — regular psychological check-ups are associated with better reintegration outcomes. This could mean check-ups help, or that more compliant/progressing residents attend more check-ups (reverse causality).

- **`post_placement_visits`** (coef ~ +0.15, p=0.081): Positive association — more post-placement monitoring visits are linked to better outcomes. This is encouraging for the organization's post-placement program.

- **`trauma_severity_score`** (coef ~ +0.13, p=0.09): Weakly positive, which seems counterintuitive. Higher-trauma cases may receive more intensive intervention (confounding). This does NOT mean more trauma leads to better outcomes — it reflects the intensity of support provided.

- **`case_category_Neglected`** (coef ~ -0.13, p=0.086): Neglected cases show a negative association with completion relative to the reference category, potentially reflecting different family dynamics and reintegration barriers.

### Confounding and Reverse Causality

- **Intervention intensity confounding:** Visits, sessions, and check-ups may increase for girls who are *already progressing* (reverse causality) or for harder cases that need more support (confounding by indication). Both make coefficient interpretation ambiguous.
- **Courses completed paradox:** The negative coefficient likely reflects a mediator/confounder relationship rather than a direct effect. Residents enrolled in more courses may be in longer-term care trajectories.

### What we can and cannot claim

**Defensible claims (hypothesis-generating):**
- Post-placement monitoring is associated with better reintegration outcomes
- Psychological check-up compliance is positively associated with completion
- Case category (family situation) meaningfully shifts baseline completion probabilities

**Claims requiring stronger evidence:**
- We cannot claim that adding more check-ups will *cause* better outcomes
- We cannot claim that course enrollment *hurts* reintegration (confounded)
- Causal claims would require randomized assignment of intervention intensity or quasi-experimental designs

## 8) Deployment Notes

### Model Artifacts
- **Model file:** `models/reintegration-drivers/model.sav` (OLS model + scaler + feature list)
- **Run log:** `models/reintegration-drivers/model.json` (append-only metadata + metrics per training run)

### Inference Pipeline
- **Nightly inference:** `ml/reintegration_drivers/infer.py` writes **one** `org_insight` row to the `ml_predictions` table (not per-resident), containing the top driver coefficients and their effect sizes.
- **Write mechanism:** `ml/utils_db.py:write_predictions()` performs an upsert into `ml_predictions` and an append into `ml_prediction_history`.
- **Model name in DB:** `model_name = 'reintegration-drivers'`, `entity_type = 'org_insight'`, `score_label = 'explainer'`
- **Metadata JSON:** Contains `top_drivers` list with feature, coefficient, p_value, ci_lower, ci_upper, and label for each; plus `adjusted_r2` and `n_observations`.

### Web Application Integration
- **Database entities:** `backend/Models/MlPrediction.cs` and `backend/Models/MlPredictionHistory.cs` define the C# entity models.
- **DbContext registration:** `backend/Data/AppDbContext.cs` line 50 registers `DbSet<MlPrediction> MlPredictions`.
- **API layer:** The backend (ASP.NET minimal API in `backend/Program.cs`) queries `MlPredictions` where `ModelName == "reintegration-drivers"` and serves the org-level insight. The top driver coefficients are displayed on the **Admin Dashboard** in the analytics/insights section, giving leadership a ranked list of which factors most strongly explain reintegration success.
- **Complementary pipeline:** This complements Pipeline 1's per-resident readiness scores -- staff see both the *who* (readiness scores) and the *why* (driver insights) in one view.

### Retraining & Monitoring
- Re-run the notebook after material data refreshes; keep `training_date` in metadata for versioning. With n=60, each new completed reintegration materially changes the coefficient estimates.
- Track whether the significant coefficients remain stable across retraining runs. If the sign or magnitude of key drivers (e.g., `psych_checkups`, `post_placement_visits`) changes substantially, investigate whether the underlying data distribution has shifted.

In [7]:
# Ch. 17 — Save artifacts


def coef_table(fit) -> list[dict]:
    rows = []
    conf = fit.conf_int()
    for name in fit.params.index:
        if name == "const":
            continue
        lo = float(conf.loc[name].iloc[0])
        hi = float(conf.loc[name].iloc[1])
        rows.append(
            {
                "feature": name,
                "coefficient": float(fit.params[name]),
                "std_err": float(fit.bse[name]),
                "p_value": float(fit.pvalues[name]),
                "ci_lower": lo,
                "ci_upper": hi,
            }
        )
    return rows


ols_rows = coef_table(ols)
logit_rows = coef_table(logit) if logit is not None else None

save_model_bundle(ols, scaler, list(X_sel.columns), logit)

meta = save_metadata(
    model_type="OLS (statsmodels) + Logit comparison",
    feature_list=list(X_sel.columns),
    train_rows=len(X_train),
    test_rows=len(X_test),
    total_rows=len(train_df),
)

metrics = save_metrics(
    adjusted_r2=float(ols.rsquared_adj),
    pseudo_r2=float(logit.prsquared) if (logit is not None and hasattr(logit, "prsquared")) else None,
    n_observations=int(len(y_train)),
    ols_coefficients=ols_rows,
    logit_coefficients=logit_rows,
)

print("Saved:")
print(" -", MODEL_REINTEGRATION_DRIVERS)
print(" -", MODEL_RUNS_REINTEGRATION_DRIVERS)
print("metadata keys:", meta.keys())
print("metrics keys:", metrics.keys())


Saved:
 - C:\Users\justi\Desktop\BYU\INTEX\intex2-1\models\reintegration-drivers\model.sav
 - C:\Users\justi\Desktop\BYU\INTEX\intex2-1\models\reintegration-drivers\model.json
metadata keys: dict_keys(['model_name', 'model_version', 'trained_at_utc', 'features', 'num_training_rows', 'num_test_rows', 'training_date', 'model_type', 'feature_list', 'train_rows', 'test_rows', 'total_rows'])
metrics keys: dict_keys(['model_name', 'model_version', 'trained_at_utc', 'features', 'num_training_rows', 'num_test_rows', 'training_date', 'model_type', 'feature_list', 'train_rows', 'test_rows', 'total_rows', 'accuracy', 'f1', 'roc_auc', 'classification_report', 'adjusted_r2', 'pseudo_r2', 'n_observations', 'ols_coefficients'])
